# 載入資料集與預處理

In [5]:
from keras.datasets import mnist

(x_train_image, y_train_label), (x_test_image, y_test_label) = mnist.load_data()

# reshape 為4維資料
x_train_image_4d = x_train_image.reshape(-1, 28, 28, 1).astype('float32')
x_test_image_4d = x_test_image.reshape(-1, 28, 28, 1).astype('float32')

# 標準化
x_train_normalize = x_train_image_4d / 255
x_test_normalize = x_test_image_4d / 255

# 標籤進行One-hot Encoding
from tensorflow.keras.utils import to_categorical

y_train_onehot = to_categorical(y_train_label)
y_test_onehot = to_categorical(y_test_label)

In [6]:
print("Train Feature:", x_train_normalize.shape)
print("Train Label:", y_train_onehot.shape)
print("Test Feature:", x_test_normalize.shape)
print("Test Label:", y_test_onehot.shape)

Train Feature: (60000, 28, 28, 1)
Train Label: (60000, 10)
Test Feature: (10000, 28, 28, 1)
Test Label: (10000, 10)


# 建立模型

In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout

model = Sequential()

# 卷積層
model.add(Conv2D(
    filters=32,
    kernel_size=(5,5),
    padding='same',
    input_shape=(28, 28, 1),
    activation='relu'))
# 池化層
model.add(MaxPooling2D(pool_size=(2,2)))

# 卷積層
model.add(Conv2D(
    filters=64,
    kernel_size=(5,5),
    padding='same',
    activation='relu'))
# 池化層
model.add(MaxPooling2D(pool_size=(2,2)))

# 平坦層
model.add(Flatten())
# 隱藏層
model.add(Dense(128, activation='relu'))
# 輸出層
model.add(Dense(10, activation='softmax'))

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 32)     │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 14, 14, 64)     │        51,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 3136)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       401,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 454,922 (1.74 MB)

 Trainable params: 454,922 (1.74 MB)

 Non-trainable params: 0 (0.00 B)

In [8]:
model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['acc'])

train_history = model.fit(
    x=x_train_normalize,
    y=y_train_onehot,
    validation_split=0.2,
    epochs=10,
    batch_size=200,
    verbose=2)

Epoch 1/10
240/240 - 5s - 22ms/step - acc: 0.9311 - loss: 0.2341 - val_acc: 0.9778 - val_loss: 0.0730
Epoch 2/10
240/240 - 3s - 12ms/step - acc: 0.9819 - loss: 0.0580 - val_acc: 0.9864 - val_loss: 0.0478
Epoch 3/10
240/240 - 2s - 8ms/step - acc: 0.9874 - loss: 0.0381 - val_acc: 0.9839 - val_loss: 0.0533
Epoch 4/10
240/240 - 2s - 8ms/step - acc: 0.9905 - loss: 0.0296 - val_acc: 0.9872 - val_loss: 0.0416
Epoch 5/10
240/240 - 2s - 8ms/step - acc: 0.9930 - loss: 0.0215 - val_acc: 0.9886 - val_loss: 0.0369
Epoch 6/10
240/240 - 2s - 8ms/step - acc: 0.9946 - loss: 0.0170 - val_acc: 0.9900 - val_loss: 0.0332
Epoch 7/10
240/240 - 2s - 9ms/step - acc: 0.9962 - loss: 0.0122 - val_acc: 0.9892 - val_loss: 0.0378
Epoch 8/10
240/240 - 2s - 8ms/step - acc: 0.9965 - loss: 0.0108 - val_acc: 0.9889 - val_loss: 0.0378
Epoch 9/10
240/240 - 2s - 8ms/step - acc: 0.9966 - loss: 0.0098 - val_acc: 0.9893 - val_loss: 0.0382
Epoch 10/10
240/240 - 2s - 8ms/step - acc: 0.9975 - loss: 0.0077 - val_acc: 0.9914 - val_

# 硬體加速

In [1]:
import os
import tensorflow as tf

def check_accelerator():
  if tf.test.gpu_device_name()!='':
    print('Default GPU Device: {}'.format(tf.test.gpu_device_name()))
    !nvidia-smi # 顯示詳細資訊

  else:
    if 'COLAB_TPU_ADDR' in os.environ:
      print('TPU')
    else:
      print('No accelerator')

check_accelerator()

Default GPU Device: /device:GPU:0
Sat Apr 18 08:28:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P0             27W /   70W |     105MiB /  15360MiB |      4%      Default |
|                                         |                        |                  N/A |
+-------------